# Olist Data Preparation and Order-Level Feature Construction

This notebook prepares the common modeling dataset for the at-placement and at-delivery classification tracks. The raw Olist data are stored across several relational tables, so the main objective is to construct a reliable one-row-per-order dataset while preserving the timing of when each feature becomes available.

The preparation steps address duplicate reviews, multi-item and multi-payment orders, missing values, and timestamp anomalies. Numeric imputation, encoding, scaling, feature selection, class balancing, and hyperparameter tuning are intentionally left to the model-specific training pipelines so that those steps are estimated using training data only.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

RAW_DIR = Path('../data/raw')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


## 1. Load the raw tables

The project uses the order, item, payment, review, product, seller, customer, geolocation, and product-category translation files from the Olist dataset. The geolocation table is loaded for possible later feature engineering but is not used in the base version of the master dataset.


In [ ]:
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
cat_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")


## 2. Parse timestamps and define the comparison cohort

The two supervised models are compared on the same completed-order cohort. Orders are therefore restricted to those with `order_status == "delivered"`.

This choice makes the at-placement and at-delivery results directly comparable on the same orders. It also narrows the interpretation of the final models: the results apply to orders that ultimately completed delivery rather than to every order initially placed.


In [ ]:
order_date_cols = [
    "order_purchase_timestamp","order_approved_at",
    "order_delivered_carrier_date","order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for c in order_date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

for c in ["review_creation_date","review_answer_timestamp"]:
    reviews[c] = pd.to_datetime(reviews[c], errors="coerce")

orders_clean = orders.loc[orders["order_status"].eq("delivered")].copy()


## 3. Resolve duplicate reviews and define the target

Some orders have more than one review record. To keep one outcome per order, review records are sorted by their review timestamps and the most recent review is retained.

The binary target is `review_bad = 1` for review scores of 1 or 2, and 0 otherwise. Review timestamps are kept only for resolving duplicate records and are not used as model predictors.


In [ ]:
reviews_clean = (
    reviews.sort_values(
        ["order_id","review_creation_date","review_answer_timestamp"],
        na_position="first"
    )
    .drop_duplicates("order_id", keep="last")
    [["order_id","review_score","review_creation_date","review_answer_timestamp"]]
    .copy()
)
reviews_clean["review_bad"] = (reviews_clean["review_score"] <= 2).astype("int8")


## 4. Aggregate item and product information to order level

The order-item table can contain several rows for the same order. Joining it directly to the order table would duplicate the target, so item-level information is summarized before the master table is created.

Order-level features include the number of items, distinct sellers, products and categories, total price and freight, and summary product dimensions. For representative product and seller fields, the highest-priced item is used.

Numeric product values are not imputed at this stage. Any remaining missing numeric values are handled later inside the training and cross-validation pipeline.


In [ ]:
products_clean = products.merge(cat_translation, on="product_category_name", how="left")
products_clean["product_category_name_english"] = (
    products_clean["product_category_name_english"].fillna("unknown")
)
products_clean["product_volume_cm3"] = (
    products_clean["product_length_cm"] *
    products_clean["product_height_cm"] *
    products_clean["product_width_cm"]
)

item_detail = items.merge(
    products_clean[[
        "product_id","product_category_name_english",
        "product_weight_g","product_length_cm","product_height_cm",
        "product_width_cm","product_volume_cm3"
    ]],
    on="product_id", how="left"
)

items_agg = item_detail.groupby("order_id").agg(
    n_items=("order_item_id","count"),
    n_distinct_sellers=("seller_id","nunique"),
    n_distinct_products=("product_id","nunique"),
    n_product_categories=("product_category_name_english","nunique"),
    total_price=("price","sum"),
    total_freight=("freight_value","sum"),
    avg_item_price=("price","mean"),
    avg_product_weight_g=("product_weight_g","mean"),
    max_product_weight_g=("product_weight_g","max"),
    avg_product_volume_cm3=("product_volume_cm3","mean"),
    max_product_volume_cm3=("product_volume_cm3","max"),
).reset_index()

primary_item = (
    item_detail.sort_values(["order_id","price","order_item_id"], ascending=[True,False,True])
    .drop_duplicates("order_id")
    [["order_id","product_id","seller_id","product_category_name_english"]]
    .rename(columns={
        "product_id":"primary_product_id",
        "seller_id":"primary_seller_id",
        "product_category_name_english":"primary_product_category"
    })
)
items_agg = items_agg.merge(primary_item, on="order_id", how="left")


## 5. Aggregate payment information

Orders may also contain multiple payment records. Payment information is therefore summarized to one row per order.

The primary payment type is defined as the payment method accounting for the largest total payment value within the order. This is more representative than selecting a payment type based only on row order.


In [ ]:
payments_agg = payments.groupby("order_id").agg(
    total_payment_value=("payment_value","sum"),
    n_payment_installments=("payment_installments","max"),
    n_payment_methods=("payment_type","nunique"),
).reset_index()

payment_by_type = payments.groupby(
    ["order_id","payment_type"], as_index=False
)["payment_value"].sum()

primary_payment = (
    payment_by_type.sort_values(
        ["order_id","payment_value","payment_type"],
        ascending=[True,False,True]
    )
    .drop_duplicates("order_id")
    [["order_id","payment_type"]]
    .rename(columns={"payment_type":"primary_payment_type"})
)
payments_agg = payments_agg.merge(primary_payment, on="order_id", how="left")


## 6. Construct the one-row-per-order master table

Customer, item, payment, seller, and review information are merged onto the delivered-order table. The `order_id` uniqueness check confirms that the aggregation steps preserve one observation per order, which is required because the prediction target is defined at order level.


In [ ]:
customers_clean = customers[[
    "customer_id","customer_unique_id","customer_zip_code_prefix",
    "customer_city","customer_state"
]].copy()

sellers_clean = sellers[[
    "seller_id","seller_zip_code_prefix","seller_city","seller_state"
]].copy().rename(columns={
    "seller_id":"primary_seller_id",
    "seller_zip_code_prefix":"primary_seller_zip_code_prefix",
    "seller_city":"primary_seller_city",
    "seller_state":"primary_seller_state"
})

master = (
    orders_clean
    .merge(customers_clean, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="inner")
    .merge(payments_agg, on="order_id", how="left")
    .merge(sellers_clean, on="primary_seller_id", how="left")
    .merge(reviews_clean, on="order_id", how="left")
)

assert master["order_id"].is_unique, "One-row-per-order rule was violated."


## 7. Engineer placement-stage and delivery-stage features

The engineered variables are separated conceptually by decision point.

Placement-stage variables describe purchase timing, the promised delivery window, freight burden, order complexity, and whether the customer and primary seller are located in the same state.

Delivery-stage variables describe actual delivery duration, lateness relative to the estimate, approval and carrier timing, shipping transit, and the number of late or early days. These variables are available only to the at-delivery model.

Negative approval-to-carrier intervals are treated as timestamp anomalies. They are flagged explicitly and the invalid duration is set to missing rather than being treated as a valid shipping interval.


In [ ]:
master["purchase_hour"] = master["order_purchase_timestamp"].dt.hour.astype("Int64")
master["purchase_day_of_week"] = master["order_purchase_timestamp"].dt.dayofweek.astype("Int64")
master["purchase_month"] = master["order_purchase_timestamp"].dt.month.astype("Int64")
master["is_weekend"] = master["purchase_day_of_week"].isin([5,6]).astype("int8")

master["promised_delivery_days"] = (
    master["order_estimated_delivery_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 86400

master["freight_to_price_ratio"] = np.where(
    master["total_price"] > 0,
    master["total_freight"] / master["total_price"],
    np.nan
)
master["avg_freight_per_item"] = np.where(
    master["n_items"] > 0,
    master["total_freight"] / master["n_items"],
    np.nan
)
master["same_customer_seller_state"] = (
    master["customer_state"].notna()
    & master["primary_seller_state"].notna()
    & master["customer_state"].eq(master["primary_seller_state"])
).astype("int8")

master["delivery_time_days"] = (
    master["order_delivered_customer_date"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 86400

master["delivery_vs_estimate_days"] = (
    master["order_delivered_customer_date"] - master["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

master["approval_delay_days"] = (
    master["order_approved_at"] - master["order_purchase_timestamp"]
).dt.total_seconds() / 86400

master["carrier_pickup_delay_days_raw"] = (
    master["order_delivered_carrier_date"] - master["order_approved_at"]
).dt.total_seconds() / 86400

master["carrier_timestamp_anomaly"] = (
    master["carrier_pickup_delay_days_raw"] < 0
).astype("int8")

master["carrier_pickup_delay_days"] = (
    master["carrier_pickup_delay_days_raw"]
    .mask(master["carrier_pickup_delay_days_raw"] < 0, np.nan)
)

master["shipping_transit_days"] = (
    master["order_delivered_customer_date"] - master["order_delivered_carrier_date"]
).dt.total_seconds() / 86400

master["delivered_late"] = (master["delivery_vs_estimate_days"] > 0).astype("int8")
master["late_days"] = master["delivery_vs_estimate_days"].clip(lower=0)
master["early_days"] = (-master["delivery_vs_estimate_days"]).clip(lower=0)


## 8. Finalize the supervised-learning cohort

Orders without an observed review cannot be used for supervised learning because the target is unavailable. Orders missing the delivery timing required for the at-delivery comparison are also removed.

Missing categorical values are grouped as `unknown`. Remaining numeric missing values are retained so that imputation can be fitted using training data rather than the full dataset.


In [ ]:
master = master.dropna(subset=["review_bad"]).copy()
master = master.dropna(subset=[
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_time_days",
    "delivery_vs_estimate_days"
]).copy()

for c in [
    "customer_state","customer_city",
    "primary_seller_state","primary_seller_city",
    "primary_payment_type","primary_product_category"
]:
    master[c] = master[c].fillna("unknown")


## 9. Freeze the shared train/test split

A stratified 80/20 split is created using `review_bad`, with `random_state=42`. The same split must be used by both the at-placement and at-delivery tracks so that their final precision, recall, and F1 scores are directly comparable.

No model-selection decision should be based on the held-out test set. Preprocessing, feature selection, class-imbalance treatment, and hyperparameter tuning should be performed on the training sample or within cross-validation.


In [ ]:
train_ids, test_ids = train_test_split(
    master["order_id"],
    test_size=0.20,
    random_state=42,
    stratify=master["review_bad"]
)

split_map = pd.DataFrame({
    "order_id": list(train_ids) + list(test_ids),
    "split": ["train"] * len(train_ids) + ["test"] * len(test_ids)
})
master = master.merge(split_map, on="order_id", how="left")

print(master["split"].value_counts())
print(master.groupby("split")["review_bad"].mean())


## 10. Modeling rules

The resulting master dataset is the common input to the team's supervised modeling work.

- Gabby and Lambert use the same frozen `split`.
- Review-derived fields are excluded from all predictor sets.
- Delivery-only variables are excluded from the at-placement model.
- IDs are retained for joins, audit checks, and case-level error analysis, but are not used directly as predictors.
- Imputation, encoding, scaling, resampling, feature selection, and tuning are fitted using training data only.
- The held-out test set is reserved for the final evaluation after the model specification has been selected.

The geolocation table and customer-history variables are not used in this base preparation notebook. They can be evaluated later as separate feature-engineering experiments if the team decides they add enough value.


## 11. Final data checks

Before saving the modeling files, a short set of checks confirms the final sample size, class balance, order-level uniqueness, split sizes, and the number of carrier timestamp anomalies.


In [ ]:
audit = pd.DataFrame({
    "check": [
        "Final modeling rows",
        "Unique order_id",
        "Negative-review rows",
        "Negative-review rate",
        "Train rows",
        "Test rows",
        "Carrier timestamp anomalies"
    ],
    "value": [
        len(master),
        master["order_id"].nunique(),
        int(master["review_bad"].sum()),
        float(master["review_bad"].mean()),
        int((master["split"] == "train").sum()),
        int((master["split"] == "test").sum()),
        int(master["carrier_timestamp_anomaly"].sum())
    ]
})

audit


## 12. Save canonical processed outputs

All reusable analytical datasets are written to `data/processed/`. This folder is the single source of truth for downstream clustering, modeling, interpretation, and Power BI export.


In [ ]:
master.to_csv(
    PROCESSED_DIR / 'master_orders_clean_v2.csv',
    index=False
)

master[['order_id', 'split', 'review_bad']].to_csv(
    PROCESSED_DIR / 'shared_train_test_split.csv',
    index=False
)

audit.to_csv(
    PROCESSED_DIR / 'data_quality_audit_v2.csv',
    index=False
)

print('Saved:', PROCESSED_DIR / 'master_orders_clean_v2.csv')
print('Saved:', PROCESSED_DIR / 'shared_train_test_split.csv')
print('Saved:', PROCESSED_DIR / 'data_quality_audit_v2.csv')
